# PoC 1: LLM → Cross-Impact Matrix → CIB Consistent Scenarios

GPT-4o fills a cross-impact matrix for R&S-relevant technology drivers. CIB algorithm filters for internally consistent scenario combinations.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv("../.env")

from openai import AzureOpenAI
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.models import Driver, DriverState, SourceStateImpacts
from src.cib import build_state_index, total_states, find_consistent_scenarios

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)
MODEL = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
print(f"Connected: {MODEL}")

## Define Technology Drivers (R&S Domain)

In [ ]:
drivers = [
    Driver(
        name="6G Development",
        description="Progress of 6G wireless standard development and deployment timelines",
        states=[
            DriverState(name="delayed", description="6G standards delayed beyond 2032, slow R&D investment"),
            DriverState(name="on-track", description="6G standards on track for ~2030, steady progress"),
            DriverState(name="accelerated", description="6G pushed ahead of schedule by geopolitical competition"),
        ],
    ),
    Driver(
        name="Spectrum Policy",
        description="Regulatory stance on radio spectrum allocation and sharing",
        states=[
            DriverState(name="restrictive", description="Regulators tighten spectrum access, more exclusive licensing"),
            DriverState(name="status-quo", description="Current spectrum policy framework remains largely unchanged"),
            DriverState(name="liberalized", description="Dynamic spectrum sharing widely adopted, open access models"),
        ],
    ),
    Driver(
        name="AI-Native Wireless",
        description="Integration of AI/ML into wireless network design and operation",
        states=[
            DriverState(name="niche", description="AI used only in narrow optimization tasks"),
            DriverState(name="mainstream", description="AI integrated into most network planning and operations"),
            DriverState(name="dominant", description="AI-native architectures replace traditional signal processing"),
        ],
    ),
    Driver(
        name="Space-Based Sensing",
        description="Proliferation of LEO satellite constellations for spectrum monitoring and comms",
        states=[
            DriverState(name="limited", description="Few constellations, mostly military/government"),
            DriverState(name="growing", description="Multiple commercial constellations, increasing coverage"),
            DriverState(name="pervasive", description="Dense LEO mesh provides global real-time RF awareness"),
        ],
    ),
    Driver(
        name="EW Demand",
        description="Demand for electronic warfare and spectrum dominance capabilities",
        states=[
            DriverState(name="stable", description="EW budgets and requirements remain at current levels"),
            DriverState(name="increasing", description="Rising geopolitical tensions drive EW investment"),
            DriverState(name="surge", description="Active conflicts create urgent EW capability demands"),
        ],
    ),
    Driver(
        name="T&M Automation",
        description="Level of automation in test and measurement workflows",
        states=[
            DriverState(name="manual", description="Most T&M workflows remain operator-driven"),
            DriverState(name="semi-auto", description="Automated data collection, human analysis"),
            DriverState(name="fully-auto", description="End-to-end automated T&M with AI-driven anomaly detection"),
        ],
    ),
    Driver(
        name="Open RAN Adoption",
        description="Market adoption of Open RAN disaggregated network architecture",
        states=[
            DriverState(name="stalled", description="Open RAN fails to gain traction beyond trials"),
            DriverState(name="gradual", description="Slow but steady adoption in greenfield deployments"),
            DriverState(name="rapid", description="Open RAN becomes dominant architecture for new deployments"),
        ],
    ),
]

n_combos = 1
for d in drivers:
    n_combos *= len(d.states)
print(f"{len(drivers)} drivers, {sum(len(d.states) for d in drivers)} total states, {n_combos} combinations")

## CIB Sanity Check: Known 2×2 Case

Before using the LLM, verify the CIB algorithm with a hand-crafted example where we know the answer.

In [ ]:
# 2 drivers, 2 states each -> 4 combinations
# Driver A: [a0, a1], Driver B: [b0, b1]
# Cross-impact: a0 promotes b0, a1 promotes b1 (and vice versa)
# -> consistent: (a0,b0) and (a1,b1)
test_drivers = [
    Driver(name="A", description="test", states=[
        DriverState(name="a0", description=""), DriverState(name="a1", description="")
    ]),
    Driver(name="B", description="test", states=[
        DriverState(name="b0", description=""), DriverState(name="b1", description="")
    ]),
]
# Matrix: rows=[a0, a1, b0, b1], cols=[a0, a1, b0, b1]
# Self-driver cells are ignored by the algorithm
test_matrix = np.array([
    #  a0  a1  b0  b1
    [  0,  0,  2, -2],  # a0 promotes b0, inhibits b1
    [  0,  0, -2,  2],  # a1 promotes b1, inhibits b0
    [  2, -2,  0,  0],  # b0 promotes a0, inhibits a1
    [ -2,  2,  0,  0],  # b1 promotes a1, inhibits a0
], dtype=float)

result = find_consistent_scenarios(test_matrix, test_drivers)
print(f"Consistent scenarios: {result}")
assert len(result) == 2
assert {"A": "a0", "B": "b0"} in result
assert {"A": "a1", "B": "b1"} in result
print("✓ CIB algorithm verified")

## LLM fills the Cross-Impact Matrix

One API call per source-state. The LLM rates how that state influences every state of every *other* driver (-3 to +3).

In [ ]:
state_index = build_state_index(drivers)
n_states = total_states(drivers)
matrix = np.zeros((n_states, n_states))

SYSTEM_PROMPT = """You are an expert in wireless communications, spectrum management, and defense technology.
You are filling a Cross-Impact Balance matrix for strategic technology foresight (horizon: 2030-2035).
For a given source driver state, rate its influence on every state of every other driver.
Scale: -3 (strongly inhibits) to +3 (strongly promotes). 0 = no significant influence.
Be specific and differentiated — avoid rating everything 0 or 1."""

for src_driver in drivers:
    for src_state in src_driver.states:
        target_info = []
        for tgt_driver in drivers:
            if tgt_driver.name == src_driver.name:
                continue
            for tgt_state in tgt_driver.states:
                target_info.append(f"- {tgt_driver.name} / {tgt_state.name}: {tgt_state.description}")

        user_msg = (
            f"Source: {src_driver.name} is in state '{src_state.name}' — {src_state.description}\n\n"
            f"Rate how this influences each of the following target states:\n"
            + "\n".join(target_info)
        )

        print(f"  → {src_driver.name} / {src_state.name}...", end=" ", flush=True)
        response = client.beta.chat.completions.parse(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg},
            ],
            response_format=SourceStateImpacts,
            temperature=0.3,
        )
        parsed = response.choices[0].message.parsed

        for impact in parsed.impacts:
            tgt_key = (impact.target_driver, impact.target_state)
            if tgt_key in state_index:
                src_idx = state_index[(src_driver.name, src_state.name)]
                tgt_idx = state_index[tgt_key]
                matrix[src_idx, tgt_idx] = impact.impact

        print("done")

print(f"\nMatrix filled: {np.count_nonzero(matrix)} non-zero entries out of {n_states**2}")

## Cross-Impact Heatmap

In [ ]:
labels = [f"{d.name}\n{s.name}" for d in drivers for s in d.states]

fig, ax = plt.subplots(figsize=(18, 14))
sns.heatmap(
    matrix, xticklabels=labels, yticklabels=labels,
    cmap="RdBu_r", center=0, vmin=-3, vmax=3,
    annot=True, fmt=".0f", linewidths=0.5, ax=ax,
)
ax.set_xlabel("Target State")
ax.set_ylabel("Source State")
ax.set_title("Cross-Impact Matrix (LLM-generated)")
plt.tight_layout()
plt.show()

## Run CIB — Find Consistent Scenarios

In [ ]:
consistent = find_consistent_scenarios(matrix, drivers)
print(f"Found {len(consistent)} consistent scenarios out of {n_combos} total\n")

df = pd.DataFrame(consistent)
df.index.name = "Scenario"
df

## Quick Scenario Narratives

Ask the LLM to give a one-paragraph narrative for each consistent scenario.

In [ ]:
from IPython.display import Markdown, display

for i, scenario in enumerate(consistent[:5]):
    desc = "\n".join(f"- **{k}**: {v}" for k, v in scenario.items())
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a strategic foresight analyst for a spectrum monitoring and T&M company (like Rohde & Schwarz). Write vivid, concrete scenario narratives."},
            {"role": "user", "content": f"Write a 3-4 sentence 'A day in the life of a spectrum monitoring operator in 2033' narrative for this scenario:\n\n{desc}"},
        ],
        temperature=0.7,
        max_tokens=300,
    )
    display(Markdown(f"### Scenario {i+1}\n{desc}\n\n> {response.choices[0].message.content}\n\n---"))